In [ ]:
import pandas as pd
import numpy as np
import random

# 读取原始CSV文件
data = pd.read_csv(r'./data/data_div_train_and_test/train_data.csv', sep=',', names=['user_id', 'item_id', 'rating'], engine='python')

# 设置随机数种子以确保可复现性
random.seed(42)

# 获取所有不同的user_id
unique_user_ids = data['user_id'].unique()

# 随机打乱user_id的顺序
random.shuffle(unique_user_ids)

# 设置要分成的子数据集数量
num_subsets = 15  # 将划分好的大训练集集分为n份

# 将user_id均匀地分成n份
num_users = len(unique_user_ids)
subset_size = num_users // num_subsets
subsets = []

for i in range(num_subsets):
    if i < num_subsets - 1:
        user_subset = unique_user_ids[i * subset_size: (i + 1) * subset_size]
    else:
        user_subset = unique_user_ids[i * subset_size:]
    
    subset = data[data['user_id'].isin(user_subset)]
    subsets.append(subset)

# 保存n个子数据集为CSV文件
for i, subset in enumerate(subsets):
    subset.to_csv(f'./data/div_shard{num_subsets}/sub_train{i+1}.csv', sep=',', header=False, index=False)  # 保存到文件subset_1.csv, subset_2.csv, 等等

print(f"训练集数据已按要求划分为{num_subsets}个子数据集并保存为CSV文件。")

# 现在处理测试集
test_data = pd.read_csv(r'./data/data_div_train_and_test/test_data.csv', sep=',', names=['user_id', 'item_id', 'rating'], engine='python')
test_subsets = []

for i, user_subset in enumerate(subsets):
    test_subset = test_data[test_data['user_id'].isin(user_subset['user_id'].unique())]
    test_subsets.append(test_subset)

# 保存n个子测试集为CSV文件
for i, test_subset in enumerate(test_subsets):
    test_subset.to_csv(f'./data/div_shard{num_subsets}/sub_test{i+1}.csv', sep=',', header=False, index=False)

print(f"测试集数据已按要求划分为{num_subsets}个子数据集并保存为CSV文件。")
